# 01 Universe Lineage Quotes

Este notebook reconstruye la genealogía del universo que terminó alimentando la corrida de quotes de producción.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq

PROJECT_ROOT = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps")
UNIVERSE_DIR = PROJECT_ROOT / "data" / "reference" / "universe_pti"

PATHS = {
    "pti_all": UNIVERSE_DIR / "tickers_all.parquet",
    "pti_active": UNIVERSE_DIR / "tickers_active.parquet",
    "pti_inactive": UNIVERSE_DIR / "tickers_inactive.parquet",
    "tickers_2005_2026": UNIVERSE_DIR / "tickers_2005_2026.parquet",
    "tickers_2005_2026_upper": UNIVERSE_DIR / "tickers_2005_2026_upper.parquet",
    "missing_1m_vs_daily": UNIVERSE_DIR / "tickers_missing_in_ohlcv_1m_vs_daily.csv",
    "missing_1m_vs_daily_lt_2b": UNIVERSE_DIR / "tickers_missing_in_ohlcv_1m_vs_daily_lt_2B.csv",
    "missing_1m_vs_daily_summary": UNIVERSE_DIR / "tickers_missing_in_ohlcv_1m_vs_daily.summary.json",
    "universe_refined_quotes": UNIVERSE_DIR / "tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet",
    "hybrid_enriched": UNIVERSE_DIR / "hybrid_enriched" / "universe_hybrid_enriched_with_financial_ranges.parquet",
}

def row_count(path: Path) -> int | None:
    if not path.exists() or path.suffix.lower() != ".parquet":
        return None
    try:
        return int(pq.ParquetFile(path).metadata.num_rows)
    except Exception:
        return None

rows = []
for name, path in PATHS.items():
    rows.append({
        "artifact": name,
        "path": str(path),
        "exists": path.exists(),
        "row_count": row_count(path),
    })

pd.DataFrame(rows)

,artifact,path,exists,row_count
0,pti_all,C:\TSIS_Data\v1\backtest_SmallCaps\data\refere...,False,NaN
1,pti_active,C:\TSIS_Data\v1\backtest_SmallCaps\data\refere...,False,NaN
2,pti_inactive,C:\TSIS_Data\v1\backtest_SmallCaps\data\refere...,False,NaN
3,tickers_2005_2026,C:\TSIS_Data\v1\backtest_SmallCaps\data\refere...,False,NaN
4,tickers_2005_2026_upper,C:\TSIS_Data\v1\backtest_SmallCaps\data\refere...,True,12468.0
5,missing_1m_vs_daily,C:\TSIS_Data\v1\backtest_SmallCaps\data\refere...,True,NaN
6,missing_1m_vs_daily_lt_2b,C:\TSIS_Data\v1\backtest_SmallCaps\data\refere...,True,NaN
7,missing_1m_vs_daily_summary,C:\TSIS_Data\v1\backtest_SmallCaps\data\refere...,True,NaN
8,universe_refined_quotes,C:\TSIS_Data\v1\backtest_SmallCaps\data\refere...,True,12133.0
9,hybrid_enriched,C:\TSIS_Data\v1\backtest_SmallCaps\data\refere...,True,15979.0


In [2]:
lineage = pd.DataFrame([
    {
        "step": 1,
        "artifact": "tickers_all.parquet",
        "origin": "build_universe_pti.py",
        "purpose": "universo PTI base por entidad",
        "evidence_level": "confirmed",
    },
    {
        "step": 2,
        "artifact": "tickers_2005_2026.parquet",
        "origin": "build_tickers_2005_2026.py",
        "purpose": "recorte temporal 2005-2026",
        "evidence_level": "confirmed",
    },
    {
        "step": 3,
        "artifact": "tickers_2005_2026_upper.parquet",
        "origin": "build_tickers_2005_2026_upper_clean.py",
        "purpose": "lista operativa por ticker único",
        "evidence_level": "confirmed",
    },
    {
        "step": 4,
        "artifact": "tickers_missing_in_ohlcv_1m_vs_daily*.{csv,json}",
        "origin": "auditorías de cobertura",
        "purpose": "detectar faltantes de 1m frente a daily",
        "evidence_level": "confirmed",
    },
    {
        "step": 5,
        "artifact": "tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet",
        "origin": "filtro target/cobertura",
        "purpose": "universo refinado usado por quotes prod",
        "evidence_level": "confirmed_as_input_inferred_as_build_logic",
    },
])
lineage

,step,artifact,origin,purpose,evidence_level
0,1,tickers_all.parquet,build_universe_pti.py,universo PTI base por entidad,confirmed
1,2,tickers_2005_2026.parquet,build_tickers_2005_2026.py,recorte temporal 2005-2026,confirmed
2,3,tickers_2005_2026_upper.parquet,build_tickers_2005_2026_upper_clean.py,lista operativa por ticker único,confirmed
3,4,"tickers_missing_in_ohlcv_1m_vs_daily*.{csv,json}",auditorías de cobertura,detectar faltantes de 1m frente a daily,confirmed
4,5,tickers_2005_2026_upper_excluding_ohlcv_1m_mis...,filtro target/cobertura,universo refinado usado por quotes prod,confirmed_as_input_inferred_as_build_logic


In [3]:
summary_path = PATHS["missing_1m_vs_daily_summary"]
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "paths_used": {
    "input_parquet": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\universe_pti\\tickers_2005_2026_upper.parquet",
    "daily_root": "D:\\ohlcv_daily",
    "minute_root": "D:\\ohlcv_1m",
    "out_parquet": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\universe_pti\\tickers_missing_in_ohlcv_1m_vs_daily.parquet",
    "out_csv": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\universe_pti\\tickers_missing_in_ohlcv_1m_vs_daily.csv"
  },
  "counts": {
    "input_unique_tickers": 12468,
    "daily_dir_tickers": 12468,
    "minute_dir_tickers": 12106,
    "missing_in_minute_vs_daily": 362,
    "missing_in_minute_vs_input": 362,
    "written_rows": 362
  },
  "sample_missing_tickers": [
    "A",
    "AA",
    "AAPL",
    "ABT",
    "ACN",
    "ADBE",
    "ADI",
    "ADM",
    "ADP",
    "ADSK",
    "AEE",
    "AEM",
    "AEP",
    "AES",
    "AFL",
    "AG",
    "AIG",
    "AKAM",
    "ALL",
    "AMAT",
    "AMD",
    "AMGN",
    "AMT",
    "A